# Recipe 04 — Metrics
> Collect and inspect operational metrics from your pipeline.

| | |
|---|---|
| **Module** | `anchor.observability.metrics` |
| **Key classes** | `InMemoryMetricsCollector`, `LoggingMetricsCollector`, `MetricPoint` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.observability.metrics import (
    InMemoryMetricsCollector,
    LoggingMetricsCollector,
    MetricPoint,
)
from anchor.observability.exporters import OTLPMetricsExporter

## 1 — InMemoryMetricsCollector

In [ ]:
collector = InMemoryMetricsCollector()
print(f"Collector: {type(collector).__name__}")

In [ ]:
# Record several metrics during pipeline execution
collector.record_metric(name="retrieval.latency_ms", value=45.2, unit="ms")
collector.record_metric(name="retrieval.items_found", value=10, unit="count")
collector.record_metric(name="reranking.latency_ms", value=12.8, unit="ms")
collector.record_metric(name="generation.tokens_used", value=1500, unit="tokens")

print("Recorded 4 metric data points")

In [ ]:
# Retrieve and inspect — each MetricPoint has name, value, unit
metrics = collector.get_metrics()
print(f"Total metrics: {len(metrics)}\n")

for m in metrics:
    print(f"  {m.name:30s}  {m.value:>8}  {m.unit}")

## 2 — LoggingMetricsCollector and OTLPMetricsExporter

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

logging_collector = LoggingMetricsCollector()
logging_collector.record_metric(name="pipeline.tokens_used", value=3500, unit="tokens")
logging_collector.record_metric(name="pipeline.latency_ms", value=230.5, unit="ms")

print("\n(Metrics logged above via LoggingMetricsCollector)")

In [ ]:
# OTLPMetricsExporter — push to Prometheus, Grafana, etc.
# Uncomment when you have an OTLP collector running:
#
# exporter = OTLPMetricsExporter(endpoint="http://localhost:4317")
# exporter.export(collector.get_metrics())

print("OTLPMetricsExporter pattern:")
print('  exporter = OTLPMetricsExporter(endpoint="http://localhost:4317")')
print('  exporter.export(collector.get_metrics())')

## 3 — Full pipeline metrics workflow

In [ ]:
# Track an entire pipeline run with one collector
pipeline_metrics = InMemoryMetricsCollector()

# Retrieval phase
pipeline_metrics.record_metric("retrieval.latency_ms", 55.0, "ms")
pipeline_metrics.record_metric("retrieval.docs_returned", 15, "count")

# Filtering phase
pipeline_metrics.record_metric("filter.docs_passed", 8, "count")
pipeline_metrics.record_metric("filter.docs_removed", 7, "count")

# Generation phase
pipeline_metrics.record_metric("generation.input_tokens", 2000, "tokens")
pipeline_metrics.record_metric("generation.output_tokens", 500, "tokens")
pipeline_metrics.record_metric("generation.latency_ms", 180.3, "ms")

print(f"Recorded {len(pipeline_metrics.get_metrics())} pipeline metrics")

In [ ]:
# Display the full pipeline summary
all_metrics = pipeline_metrics.get_metrics()
print(f"Pipeline metrics collected: {len(all_metrics)}\n")
for m in all_metrics:
    print(f"  {m.name:35s}  {m.value:>8}  {m.unit}")

In [ ]:
# Filter metrics by unit type
latency_metrics = [m for m in all_metrics if m.unit == "ms"]
token_metrics = [m for m in all_metrics if m.unit == "tokens"]
count_metrics = [m for m in all_metrics if m.unit == "count"]

print(f"Latency metrics : {len(latency_metrics)}")
print(f"Token metrics   : {len(token_metrics)}")
print(f"Count metrics   : {len(count_metrics)}")

## Key Takeaways
- `InMemoryMetricsCollector` stores data points for programmatic access.
- `LoggingMetricsCollector` integrates with Python's logging framework.
- `OTLPMetricsExporter` pushes metrics to Prometheus, Grafana, and similar backends.
- Record metrics at each pipeline phase for full operational visibility.

**Back to:** [Observability Overview](../README.md)